In [1]:
import shutil
from pathlib import Path
import cv2
import torch
from torchmetrics.functional.image import structural_similarity_index_measure as tm_ssim
from tqdm.auto import tqdm

def deduplicate_folder_torchmetrics(
    source_folder: str,
    output_folder: str,
    threshold: float = 0.95,
    compare_size: tuple = (64, 64),
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    src_path = Path(source_folder)
    out_path = Path(output_folder)
    out_path.mkdir(parents=True, exist_ok=True)
    
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
    image_paths = sorted([p for p in src_path.iterdir() if p.suffix.lower() in IMG_EXTS])
    
    if not image_paths:
        print("No images found.")
        return

    print(f"Loading {len(image_paths)} images...")
    imgs = []
    valid_paths = []
    for p in tqdm(image_paths, desc="Loading images"):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            imgs.append(cv2.resize(img, compare_size))
            valid_paths.append(p)
            
    # Normalize images to [0.0, 1.0] for standard torchmetrics input
    # Shape: (N, 1, H, W)
    all_tensors = (torch.tensor(imgs, dtype=torch.float32).unsqueeze(1) / 255.0).to(device)

    remaining_indices = list(range(len(valid_paths)))
    unique_indices = []
    
    pbar = tqdm(total=len(valid_paths), desc="Deduplicating via TorchMetrics")
    
    while remaining_indices:
        ref_idx = remaining_indices.pop(0)
        unique_indices.append(ref_idx)
        pbar.update(1)
        
        if not remaining_indices:
            break
            
        ref_t = all_tensors[ref_idx : ref_idx + 1]       # (1, 1, H, W)
        cand_t = all_tensors[remaining_indices]           # (K, 1, H, W)
        
        # Broadcast reference across all remaining candidates: (K, 1, H, W)
        ref_expanded = ref_t.expand_as(cand_t)
        
        with torch.no_grad():
            # Official torchmetrics SSIM call directly on GPU
            scores = tm_ssim(ref_expanded, cand_t, reduction="none", data_range=1.0)
            # Flatten to 1D array of scores: (K,)
            scores = scores.view(scores.size(0), -1).mean(dim=1)
            
        keep_mask = (scores < threshold).cpu().numpy()
        discarded_count = len(remaining_indices) - keep_mask.sum()
        pbar.update(int(discarded_count))
        
        remaining_indices = [idx for idx, keep in zip(remaining_indices, keep_mask) if keep]

    pbar.close()
    
    unique_paths = [valid_paths[i] for i in unique_indices]
    print(f"\nOriginal: {len(valid_paths)} | Unique: {len(unique_paths)} | Duplicates removed: {len(valid_paths) - len(unique_paths)}")
    print(f"Copying to {output_folder} ...")
    for p in tqdm(unique_paths, desc="Copying"):
        shutil.copy2(p, out_path / p.name)
        
    print("Done!")
    return unique_paths


/home/filelele/miniconda3/envs/ars-trg-sp1_mini-project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
source_folder = "/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40"
dest_folder = "/home/filelele/personal_project/belt-conveyor-object-count/dataset/deduplicated_cropped_frames/y40"
deduplicate_folder_torchmetrics(source_folder, dest_folder, 0.7)

Loading 1187 images...


Deduplicating via TorchMetrics: 100%|██████████| 1187/1187 [00:00<00:00, 3371.81it/s]



Original: 1187 | Unique: 80 | Duplicates removed: 1107
Copying to /home/filelele/personal_project/belt-conveyor-object-count/dataset/deduplicated_cropped_frames/y40 ...


Copying: 100%|██████████| 80/80 [00:00<00:00, 8581.26it/s]

Done!


[PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000000.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000023.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000034.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000045.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000062.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000077.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000089.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyor-object-count/dataset/cropped_frames/y40/frame_000122.jpg'),
 PosixPath('/home/filelele/personal_project/belt-conveyo